# Notebook 03 — Idea → Expression Demo (No WQB)

**Goal**: Demonstrate the full idea-generation → synthesis → validation pipeline
**without** calling the WQB simulation API. This lets you iterate on the LLM
prompt quality cheaply.

Pipeline:
```
Research direction
  → IdeaAgent (RAG + LLM) → N hypotheses
  → ExprSynthAgent (LLM) → K×N FASTEXPR expressions
  → ExprValidator → filter invalid
  → NoveltyScorer → novelty scores
  → Display results
```

**Requires**: Notebooks 01 & 02 completed

In [1]:
import asyncio
import os
from pathlib import Path

import nest_asyncio
nest_asyncio.apply()

import litellm
litellm.drop_params = True  # provider-specific unsupported params are silently dropped

from alpha_agent.config import settings
from alpha_agent.knowledge.vector_store import VectorStore
from alpha_agent.knowledge.alpha_memory import AlphaMemory
from alpha_agent.data.operator_kb import OperatorKB

store = VectorStore()

# If the main DuckDB file is locked by another notebook/kernel,
# fall back to a per-process DB file for this demo notebook.
try:
    memory = AlphaMemory()
except Exception as e:
    fallback_db = settings.duckdb_path.parent / f"alpha_memory_nb03_{os.getpid()}.db"
    print(f"[Notebook03] AlphaMemory lock detected: {e}")
    print(f"[Notebook03] Falling back to: {fallback_db}")
    memory = AlphaMemory(db_path=Path(fallback_db))

kb = OperatorKB()

print("Loaded: store, memory, operator KB")
print(f"  Alpha memory path: {getattr(memory, '_path', 'unknown')}")
print(f"  Alpha memory: {memory.stats()}")

/Users/weiyanguang/vscodeprojects/WorldQuant-Brain-Alpha/myenv/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


Loaded: store, memory, operator KB
  Alpha memory path: /Users/weiyanguang/vscodeprojects/WorldQuant-Brain-Alpha/data/alpha_memory.db
  Alpha memory: {'total': 10, 'qualified': 0, 'pass_rate': 0.0}


## 1. Configure the research direction

In [2]:
# ← Change these to explore different directions
DIRECTION = "cross-sectional momentum with volume-adjusted signal strength"
DATASET   = "pv1"
N_IDEAS   = 5
K_VARIANTS = 3

## 2. Generate factor hypotheses via IdeaAgent

In [3]:
import importlib

from alpha_agent.engine import idea_agent as idea_agent_module
importlib.reload(idea_agent_module)
IdeaAgent = idea_agent_module.IdeaAgent

idea_agent = IdeaAgent(store, memory)

ideas = asyncio.run(
    idea_agent.generate_ideas(
        direction=DIRECTION,
        dataset=DATASET,
        n=N_IDEAS,
        explore_exploit_bias=0.3,  # lean toward exploration
    )
)

print(f"Generated {len(ideas)} hypotheses:\n")
for i, idea in enumerate(ideas, 1):
    print(f"[{i}] {idea.get('hypothesis', '')}")
    print(f"    Fields: {idea.get('candidate_fields', [])}")
    print(f"    Ops:    {idea.get('candidate_operators', [])}")
    print(f"    Risk:   {idea.get('risk', '')}")
    print(f"    Novel:  {idea.get('novelty_angle', '')}")
    print()

Generated 5 hypotheses:

[1] Cross-sectional momentum where signal strength is weighted by recent volume z-score relative to longer-term average, emphasizing moves with unusual volume participation.
    Fields: ['returns', 'volume']
    Ops:    ['ts_sum', 'rank', 'demean', 'vector_neut']
    Risk:   High turnover if using short windows for volume z-score; could be crowded in high-volume momentum stocks.
    Novel:  Volume adjustment is dynamic (z-score based) rather than static multiplication, emphasizing relative volume spikes within each stock's own history.

[2] Cap-neutralized momentum signal where returns are orthogonalized to market capitalization, then scaled by normalized volume rank.
    Fields: ['returns', 'volume', 'cap']
    Ops:    ['vector_neut', 'rank', 'ts_sum', 'demean']
    Risk:   Data sparsity for extreme small caps; neutralization may reduce signal power if momentum is legitimately stronger in certain cap segments.
    Novel:  Explicit cap neutralization before vol

## 3. Synthesize FASTEXPR expressions

In [4]:
from alpha_agent.engine.expr_synth_agent import ExprSynthAgent

# Use field IDs from vector store as known fields
from alpha_agent.knowledge.vector_store import COLLECTION_FIELDS
field_docs = store._col(COLLECTION_FIELDS).get(include=['metadatas'])
known_fields = {m['field_id'] for m in field_docs['metadatas'] if m.get('field_id')}
known_fields.update({'close', 'open', 'high', 'low', 'volume', 'returns', 'vwap',
                     'turnover', 'sharesout', 'cap'})
print(f"Known field count: {len(known_fields)}")

synth = ExprSynthAgent(kb, memory, known_fields=known_fields)
all_fields_list = list(known_fields)

all_expressions = {}

for idea in ideas:
    exprs = asyncio.run(
        synth.synthesize(idea, all_fields=all_fields_list, k=K_VARIANTS)
    )
    all_expressions[idea['id']] = exprs
    print(f"[{idea['id']}] {len(exprs)} expressions generated")
    for e in exprs:
        print(f"   {e}")
    print()

Known field count: 363
[vol_scaled_mom_zscore] 3 expressions generated
   rank(ts_sum(returns, 20)) * zscore(adv20 / ts_mean(adv20, 60))
   rank(ts_sum(returns, 10)) * (adv20 - ts_mean(adv20, 30)) / ts_std_dev(adv20, 30)
   rank(demean(ts_sum(returns, 15))) * (adv20 / ts_mean(adv20, 40) - 1)

[cap_neutral_vol_mom] 3 expressions generated
   vector_neut(ts_sum(returns, 20), enterprise_value) * rank(adv20)
   vector_neut(ts_mean(returns, 60), enterprise_value) * rank(adv20)
   vector_neut(ts_decay_linear(returns, 20), enterprise_value) * rank(adv20)

[volume_trend_momentum] 3 expressions generated
   rank(ts_sum(returns, 10)) * ts_corr(adv20, ts_sum(1, 10), 5)
   group_rank(ts_sum(returns, 5), fnd6_cisecgl) * ts_corr(adv20, ts_sum(1, 20), 10)
   zscore(ts_sum(returns, 15)) * if_else(ts_corr(adv20, ts_sum(1, 15), 8) > 0, ts_corr(adv20, ts_sum(1, 15), 8), 0)

[sector_rel_vol_mom] 3 expressions generated
   group_neutralize(ts_sum(returns, 20), group='sector') * group_rank(adv20, group='sec

## 4. Validate expressions locally

In [5]:
from alpha_agent.engine.validator import ExprValidator

validator = ExprValidator(kb, known_fields=known_fields)

flat_exprs = [e for exprs in all_expressions.values() for e in exprs]

valid_exprs = []
invalid_exprs = []

print(f"Validating {len(flat_exprs)} expressions...\n")
for expr in flat_exprs:
    result = validator.validate(expr)
    status = "✓ PASS" if result.ok else "✗ FAIL"
    print(f"{status}: {expr[:80]}")
    if result.errors:
        for e in result.errors:
            print(f"         ERROR: {e}")
    if result.warnings:
        for w in result.warnings:
            print(f"         WARN:  {w}")
    if result.ok:
        valid_exprs.append(expr)
    else:
        invalid_exprs.append(expr)

print(f"\nSummary: {len(valid_exprs)} valid, {len(invalid_exprs)} invalid")

Validating 15 expressions...

✓ PASS: rank(ts_sum(returns, 20)) * zscore(adv20 / ts_mean(adv20, 60))
✓ PASS: rank(ts_sum(returns, 10)) * (adv20 - ts_mean(adv20, 30)) / ts_std_dev(adv20, 30)
✓ PASS: rank(demean(ts_sum(returns, 15))) * (adv20 / ts_mean(adv20, 40) - 1)
✓ PASS: vector_neut(ts_sum(returns, 20), enterprise_value) * rank(adv20)
✓ PASS: vector_neut(ts_mean(returns, 60), enterprise_value) * rank(adv20)
✓ PASS: vector_neut(ts_decay_linear(returns, 20), enterprise_value) * rank(adv20)
✓ PASS: rank(ts_sum(returns, 10)) * ts_corr(adv20, ts_sum(1, 10), 5)
✓ PASS: group_rank(ts_sum(returns, 5), fnd6_cisecgl) * ts_corr(adv20, ts_sum(1, 20), 10)
✓ PASS: zscore(ts_sum(returns, 15)) * if_else(ts_corr(adv20, ts_sum(1, 15), 8) > 0, ts_c
✓ PASS: group_neutralize(ts_sum(returns, 20), group='sector') * group_rank(adv20, group=
         WARN:  Fields not in known set: ['group']
✓ PASS: group_neutralize(ts_sum(returns, 5) - ts_sum(returns, 60), group='sector') * gro
         WARN:  Fields not i

## 5. Novelty scoring

In [6]:
from alpha_agent.search.novelty import NoveltyScorer

novelty = NoveltyScorer(store, memory)
scores = novelty.score_batch(valid_exprs)

print("Novelty scores (higher = more novel):\n")
for expr, score in sorted(zip(valid_exprs, scores), key=lambda x: -x[1]):
    bar = '█' * int(score * 20)
    print(f"  {score:.3f} {bar}")
    print(f"         {expr[:90]}")
    print()

Novelty scores (higher = more novel):

  0.549 ██████████
         ts_sum(returns, 20) / (ts_corr(returns, adv20, 20) + 1.1)

  0.415 ████████
         group_neutralize(ts_decay_linear(returns, 30), group='sector') * group_rank(delta(adv20, 5

  0.401 ████████
         zscore(ts_sum(returns, 15)) * if_else(ts_corr(adv20, ts_sum(1, 15), 8) > 0, ts_corr(adv20,

  0.386 ███████
         group_neutralize(ts_sum(returns, 5) - ts_sum(returns, 60), group='sector') * group_rank(ts

  0.381 ███████
         group_rank(ts_sum(returns, 5), fnd6_cisecgl) * ts_corr(adv20, ts_sum(1, 20), 10)

  0.369 ███████
         vector_neut(ts_decay_linear(returns, 20), enterprise_value) * rank(adv20)

  0.367 ███████
         group_neutralize(ts_sum(returns, 20), group='sector') * group_rank(adv20, group='sector')

  0.359 ███████
         vector_neut(ts_sum(returns, 20), enterprise_value) * rank(adv20)

  0.356 ███████
         rank(ts_sum(returns, 10)) * ts_corr(adv20, ts_sum(1, 10), 5)

  0.348 ██████
     

In [7]:
# Filter by novelty threshold
from alpha_agent.config import settings

novel_exprs = novelty.filter_novel(valid_exprs, threshold=settings.novelty_score_min)
print(f"After novelty filter (threshold={settings.novelty_score_min}):")
print(f"  {len(novel_exprs)} / {len(valid_exprs)} expressions pass\n")
for e in novel_exprs:
    print(f"  → {e}")

After novelty filter (threshold=0.3):
  11 / 15 expressions pass

  → vector_neut(ts_sum(returns, 20), enterprise_value) * rank(adv20)
  → vector_neut(ts_decay_linear(returns, 20), enterprise_value) * rank(adv20)
  → rank(ts_sum(returns, 10)) * ts_corr(adv20, ts_sum(1, 10), 5)
  → group_rank(ts_sum(returns, 5), fnd6_cisecgl) * ts_corr(adv20, ts_sum(1, 20), 10)
  → zscore(ts_sum(returns, 15)) * if_else(ts_corr(adv20, ts_sum(1, 15), 8) > 0, ts_corr(adv20, ts_sum(1, 15), 8), 0)
  → group_neutralize(ts_sum(returns, 20), group='sector') * group_rank(adv20, group='sector')
  → group_neutralize(ts_sum(returns, 5) - ts_sum(returns, 60), group='sector') * group_rank(ts_mean(adv20, 10), group='sector')
  → group_neutralize(ts_decay_linear(returns, 30), group='sector') * group_rank(delta(adv20, 5), group='sector')
  → ts_sum(returns, 20) / (ts_corr(returns, adv20, 20) + 1.1)
  → rank(ts_sum(returns, 10)) / rank(ts_corr(returns, adv20, 10))
  → rank(if_else(ts_corr(returns, adv20, 5) > 0, ts_sum(r

In [8]:
# Optional: release DB/file handles before switching notebooks
import gc

for obj_name in ["memory", "registry", "store", "pi"]:
    obj = globals().get(obj_name)
    close_fn = getattr(obj, "close", None)
    if callable(close_fn):
        try:
            close_fn()
            print(f"Closed: {obj_name}")
        except Exception as e:
            print(f"Failed to close {obj_name}: {e}")

for obj_name in ["memory", "registry", "store", "pi"]:
    if obj_name in globals():
        del globals()[obj_name]

gc.collect()
print("Released DB/file handles and cleared related globals.")

Closed: memory
Released DB/file handles and cleared related globals.


## ✅ Notebook 03 Complete

You've seen the full idea → expression → validate → novelty pipeline.
No WQB API calls were made — all results are from the LLM + local checks.

The novel expressions above are ready for WQB simulation in Notebook 04.